In [4]:
# %% [markdown]
# analysis01.ipynb — Unstructured Variable Selection
# Aggregation: locationId × month (YYYY-MM)
#
# Pipeline:
#   load → preprocess → aggregate → UVS generate → initialize → fit → inspect

# %% CELL 1 — Project root & sys.path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection
SRC_DIR: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/src


In [ ]:
# %% CELL 2 — Imports
import pandas as pd
import numpy as np

from bayesian_selection.unstructured import UnstructureVariableSelection
from bayesian_selection.review_preprocess import (
    TimeBinningConfig,
    minimal_uves_input_quarterly
)

from bayesian_selection.word.word_corpus_cleaner import clean_word_corpus


In [6]:
# %% CELL 3 — Paths
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "251205_ParisHotel_comment_score.csv"
OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(exist_ok=True)

WORD_CORPUS_XLSX = OUT_DIR / "word_corpus.xlsx"
WORD_CORPUS_CLEANED = OUT_DIR / "word_corpus_cleaned.xlsx"

print("RAW_PATH:", RAW_PATH)
print("OUT_DIR:", OUT_DIR)


RAW_PATH: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/raw/251205_ParisHotel_comment_score.csv
OUT_DIR: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output


In [8]:
# %% CELL 4 — Load raw data
df_raw = pd.read_csv(RAW_PATH)
print("Raw shape:", df_raw.shape)

df_raw.columns

Raw shape: (103885, 35)


Index(['id', 'status', 'reviewDetailPageWrapper', 'location', 'createdDate',
       'publishedDate', 'userProfile', 'rating', 'publishPlatform', 'title',
       'language', 'originalLanguage', 'text', 'username',
       'connectionToSubject', 'mgmtResponse', 'locationId', 'helpfulVotes',
       'labels', 'mcid', 'attribution', 'translationType', 'mtProviderId',
       'alertStatus', 'roomTip', 'tripInfo', 'additionalRatings', 'photos',
       'content', 'corpus', 'Mobility', 'Hospitality', 'PreRegister',
       'CustomerService', 'Hygiene'],
      dtype='object')

In [ ]:
# meta_data = pd.read_csv('../data/raw/251209_FranceHotel_meta.csv')
# meta_data

,rank,__typename,locationType,locationId,snippet,mentionCount,rentalInformationV2,details,Search,name,location,rating_mean,rating_count,typename,url,country
0,0,SERP_LocationResult,ACCOMMODATION,233164,NaN,1.0,NaN,{'defaultUrl': '/Hotel_Review-g196535-d233164-...,"Hotel Manoir de Lan-Kerellec, Trébeurden",Manoir De Lan Kerellec,"Trebeurden, Brittany, France",4.7,906,Hotel,/Hotel_Review-g196535-d233164-Reviews-Manoir_D...,France
1,1,SERP_LocationResult,ACCOMMODATION,13819420,NaN,NaN,NaN,{'defaultUrl': '/Hotel_Review-g303944-d1381942...,"Hotel Manoir de Lan-Kerellec, Trébeurden",Manoir Des Arts Hotel,"Hai Phong, Vietnam",4.9,682,Hotel,/Hotel_Review-g303944-d13819420-Reviews-Manoir...,Vietnam
2,2,SERP_LocationResult,ACCOMMODATION,595599,NaN,NaN,NaN,{'defaultUrl': '/Hotel_Review-g675418-d595599-...,"Hotel Manoir de Lan-Kerellec, Trébeurden",Hotel Manoir de Restigne,"Restigne, Centre-Val de Loire, France",3.9,76,Hotel,/Hotel_Review-g675418-d595599-Reviews-Hotel_Ma...,France
3,3,SERP_LocationResult,ACCOMMODATION,325450,NaN,NaN,NaN,{'defaultUrl': '/Hotel_Review-g425021-d325450-...,"Hotel Manoir de Lan-Kerellec, Trébeurden",Hôtel Manoir de Rigourdaine,"Plouer sur Rance, Brittany, France",4.7,243,Hotel,/Hotel_Review-g425021-d325450-Reviews-Hotel_Ma...,France
4,4,SERP_LocationResult,ACCOMMODATION,586180,NaN,NaN,NaN,{'defaultUrl': '/Hotel_Review-g227608-d586180-...,"Hotel Manoir de Lan-Kerellec, Trébeurden",Manoir Hotel,"Le Touquet – Paris-Plage, Hauts-de-France, France",4.0,437,Hotel,/Hotel_Review-g227608-d586180-Reviews-Manoir_H...,France
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,5,SERP_LocationResult,ACCOMMODATION,229064,NaN,NaN,NaN,{'defaultUrl': '/Hotel_Review-g194863-d229064-...,"Hotel Hôtel du Cap-Eden-Roc, Antibes",Hotel Eden Roc,"Positano, Amalfi Coast, Italy",4.8,2239,Hotel,/Hotel_Review-g194863-d229064-Reviews-Hotel_Ed...,Italy
1886,6,SERP_LocationResult,ACCOMMODATION,3569813,NaN,1.0,NaN,{'defaultUrl': '/Hotel_Review-g16884615-d35698...,"Hotel Hôtel du Cap-Eden-Roc, Antibes",Eden Roc Cap Cana,"Cap Cana, Dominican Republic",4.8,2178,Hotel,/Hotel_Review-g16884615-d3569813-Reviews-Eden_...,Dominican Republic
1887,7,SERP_LocationResult,ACCOMMODATION,8006211,NaN,NaN,NaN,{'defaultUrl': '/Hotel_Review-g297661-d8006211...,"Hotel Hôtel du Cap-Eden-Roc, Antibes",Hotel Eden Roc,"Bhubaneswar, India",3.9,30,Hotel,/Hotel_Review-g297661-d8006211-Reviews-Hotel_E...,India
1888,8,SERP_LocationResult,ACCOMMODATION,1539178,NaN,1.0,NaN,{'defaultUrl': '/Hotel_Review-g187217-d1539178...,"Hotel Hôtel du Cap-Eden-Roc, Antibes",Cap D'antibes Beach Hotel,"Antibes, French Riviera - Cote d'Azur, France",4.3,413,Hotel,/Hotel_Review-g187217-d1539178-Reviews-Cap_D_a...,France


In [5]:
# %% CELL 5 — Text cleanup + aggregation prep (CRITICAL CELL)

# 0) Remove digits from title & text
#    (ratings, years, room numbers, prices → noise for topic discovery)
df_raw["title"] = (
    df_raw["title"]
    .fillna("")
    .astype(str)
    .str.replace(r"\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_raw["text"] = (
    df_raw["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [6]:
# %% CELL 6 — OPTIONAL: extract tripType / stayMonth from tripInfo
# Only needed if tripInfo is a stringified dict

import ast

def _safe_parse_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str) and x.strip().startswith("{"):
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}
    return {}

if "tripInfo" in df_raw.columns:
    trip = df_raw["tripInfo"].apply(_safe_parse_dict)
    df_raw["tripType"] = trip.apply(lambda d: d.get("tripType"))
    df_raw["stayMonth"] = trip.apply(
        lambda d: str(d.get("stayDate"))[:7] if d.get("stayDate") else None
    )


In [7]:
df_raw.columns

Index(['id', 'status', 'reviewDetailPageWrapper', 'location', 'createdDate',
       'publishedDate', 'userProfile', 'rating', 'publishPlatform', 'title',
       'language', 'originalLanguage', 'text', 'username',
       'connectionToSubject', 'mgmtResponse', 'locationId', 'helpfulVotes',
       'labels', 'mcid', 'attribution', 'translationType', 'mtProviderId',
       'alertStatus', 'roomTip', 'tripInfo', 'additionalRatings', 'photos',
       'content', 'corpus', 'Mobility', 'Hospitality', 'PreRegister',
       'CustomerService', 'Hygiene', 'tripType', 'stayMonth'],
      dtype='object')

In [8]:
# %% CELL 7 — Build UVS-ready dataframe
# agg_id = locationId__YYYY-MM
# review_text = [tags] + title + text

time_cfg = TimeBinningConfig(
    date_col="createdDate",
    freq="M",  # monthly binning
)

df_raw["locationId"] = (
    pd.to_numeric(df_raw["locationId"], errors="coerce")
    .astype("Int64")   # keeps NA safely
    .astype(str)
)

df = minimal_uves_input_quarterly(
    df_raw,
    location_col="locationId",
    rating_col="rating",
    title_col="title",
    text_col="text",
    date_col="createdDate",
    include_tags=False,   # experience-framing tags
)

print("UVS-ready shape:", df.shape)
df.head(3)


UVS-ready shape: (103885, 5)


,agg_id,rating,review_text,locationId,time_bin
0,23688737__2025Q2,5.0,Wonderful -star experience in the st. Perfect ...,23688737,2025Q2
1,23688737__2025Q4,5.0,Madame Reve ALL THE WAY!!!. We decided to go t...,23688737,2025Q4
2,23688737__2025Q4,3.0,Overpriced. My partner and I stayed in a Delux...,23688737,2025Q4


In [9]:
# %% CELL 8 — Initialize Unstructured Variable Selection
selector = UnstructureVariableSelection(
    data=df,
    review_col="review_text",
    rating_col="rating",
    aggregate_col="agg_id",
    word_corpus_file=None,
    dist_type="correlate",   # interpretable co-occurrence
    track=True,
)

# Generate frequency matrix + mean rating per aggregate unit
_ = selector.generate(top_n=300)

300 of words are selected


In [10]:
selector.export_word_corpus(WORD_CORPUS_XLSX)
summary = clean_word_corpus(WORD_CORPUS_XLSX)

[word_corpus_cleaner] Input file : /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output/word_corpus.xlsx
[word_corpus_cleaner] Output file: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output/word_corpus_cleaned.xlsx

[word_corpus_cleaner] Summary
  Token column detected        : corpus
  Rows (raw → final)           : 300 → 217
  Obvious-noise rows dropped   : 1
  Model-based rows dropped     : 82
  Unique tokens (before → after): 300 → 217
  Unique tokens sent to API    : 299
  Row-level token corrections  : 0
  Rows with lemma filled       : 6
  Saved cleaned workbook       : /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output/word_corpus_cleaned.xlsx


In [11]:
# %% CELL 8.1 — Re-initialize Unstructured Variable Selection

selector = UnstructureVariableSelection(
    data=df,
    review_col="review_text",
    rating_col="rating",
    aggregate_col="agg_id",
    word_corpus_file=WORD_CORPUS_CLEANED,
    dist_type="correlate",   # interpretable co-occurrence
    track=True,
)

# Generate frequency matrix + mean rating per aggregate unit
_ = selector.generate()

212 of words are selected


In [12]:
# ---- START SMALL for debugging ----
SEGMENTS = 2       # aka segment_no (latent segments K)
ITER     = 1000    # MCMC iterations
BURNIN   = 1000    # burn-in draws / default is 2000
STEP     = 1       # save every STEP draws
CUTOFF   = 0.70    # Correlate: cutoff = 0.7 / Cooccur: cutoff = 0.3

selector.initialize(
    segment_no=SEGMENTS,
    iteration=ITER,
    burnin=BURNIN,
    cutoff=CUTOFF,
    step=STEP
)
selector.fit()

  0%|          | 0/2000 [00:00<?, ?it/s]

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning:

invalid value encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning:

invalid value encountered in divide



In [16]:
selector.visualize_topic_seg(0, k = 3, cut = 0.95).update_layout(width=1200, height=600, xaxis_tickangle = 45)

In [17]:
selector.visualize_topic_seg(1, k = 3, cut = 0.95).update_layout(width=1200, height=600, xaxis_tickangle = 45)

In [ ]:
selector.find_group_members(seg_no = 0, k = 3)

{np.float64(117.0): Index(['feel', 'make', 'go', 'review', 'book', 'experience', 'disappoint',
        'hour', 'star', 'money', 'customer', 'call', 'reservation', 'central',
        'city', 'easy', 'value', 'accommodate', 'modern', 'happy', 'appreciate',
        'high', 'old', 'bathroom', 'look', 'light', 'people', 'outside',
        'large', 'shower', 'nice', 'bed', 'good', 'size'],
       dtype='object', name='corpus'),
 np.float64(118.0): Index(['clean', 'station', 'cozy', 'locate', 'pleasant', 'comfort', 'quiet',
        'friendly', 'great', 'helpful', 'welcome', 'excellent', 'spacious'],
       dtype='object', name='corpus'),
 np.float64(115.0): Index(['attentive', 'professional', 'exceptional', 'best', 'special',
        'delicious', 'perfect', 'fantastic', 'love', 'amaze', 'enjoy', 'superb',
        'beauty', 'wonderful'],
       dtype='object', name='corpus')}

In [26]:
from pandas import Index

cluster_group = {'preregister': Index(['feel', 'make', 'go', 'review', 'book', 'experience', 'disappoint',
        'hour', 'star', 'money', 'customer', 'call', 'reservation'],
       dtype='object', name='corpus'),
       'location': Index(['central',
        'city', 'easy', 'value', 'accommodate', 'modern',],
       dtype='object', name='corpus'),
    'amenity': Index([ 'happy', 'appreciate',
        'high', 'old', 'bathroom', 'look', 'light', 'people', 'outside',
        'large', 'shower', 'nice', 'bed', 'good', 'size'],
       dtype='object', name='corpus'),
 'service': Index(['clean', 'station', 'cozy', 'locate', 'pleasant', 'comfort', 'quiet',
        'friendly', 'great', 'helpful', 'welcome', 'excellent', 'spacious'],
       dtype='object', name='corpus'),
}

In [ ]:
from bayesian_selection.word.cluster import WordClusterAnalyzer

In [31]:
analyzer = WordClusterAnalyzer(
    model = selector,
    seg_no = 0,
    # k = 10000,  
    cluster_label = cluster_group,
    mode = 'PCA' # "CFA", 'calculate
).fit()

In [39]:
result_df = analyzer.transform(selector)

In [42]:
result_df['locationId'] = result_df['agg_id'].str.split('__').str[0]
result_df['quarter'] = result_df['agg_id'].str.split('__').str[1]

In [47]:
selector.find_membership()[selector.find_membership() == 1]

array([1, 1, 1, ..., 1, 1, 1], shape=(8536,))

In [38]:
corr_structure = selector.find_correlation_by_seg(0, cut = 0.9)
corr_structure['coef'] = corr_structure['coef'].apply(lambda x: round(x, 2))

for cluster in cluster_group:
    sub_structure = corr_structure
    sub_structure = sub_structure.loc[cluster_group[cluster], :].reset_index()
    sub_structure['report'] = sub_structure['corpus'] + " (" + sub_structure['coef'].astype(str) + ")"

    print(', '.join(sub_structure['corpus']))
    print(', '.join(sub_structure['coef'].astype(str)))

feel, make, go, review, book, experience, disappoint, hour, star, money, customer, call, reservation
-0.32, -0.12, -0.14, -0.21, -0.11, -0.2, -0.26, -0.21, -0.14, -0.15, -0.19, -0.23, -0.24
central, city, easy, value, accommodate, modern
0.56, 0.23, 0.46, 0.35, 1.35, 0.3
happy, appreciate, high, old, bathroom, look, light, people, outside, large, shower, nice, bed, good, size
0.84, 0.59, -0.34, -0.21, -0.18, -0.29, -0.24, -0.18, -0.38, 0.29, -0.11, 0.51, -0.14, 0.22, 0.32
clean, station, cozy, locate, pleasant, comfort, quiet, friendly, great, helpful, welcome, excellent, spacious
0.18, 0.25, 0.53, 0.62, 0.54, 0.51, 0.42, 0.26, 0.49, 0.26, 0.4, 0.86, 0.32


In [1]:
import pandas as pd 

pd.read_csv(RAW_PATH)

NameError: name 'RAW_PATH' is not defined